In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

In [8]:
load_dotenv()

True

In [3]:
pdf_link = "./data/visao-estereo-rev.pdf"

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model="gpt-3.5-turbo")


In [4]:
loader = PyPDFLoader(pdf_link, extract_images=False)
pages = loader.load_and_split()

len(pages)

16

In [5]:
# Separar em chunks

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=4000,
    chunk_overlap=20,
    length_function=len,
    add_start_index=True
)

chunks = text_splitter.split_documents(pages)

In [6]:
# Salvar os chunks em um vectorDB

vectordb = Chroma(embedding_function=embeddings,persist_directory="naiveDB")

/tmp/ipykernel_11619/3555093956.py:3: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectordb = Chroma(embedding_function=embeddings,persist_directory="naiveDB")


In [7]:
# Carregar  o DB

naive_retriever = vectordb.as_retriever(search_kwargs={"k":10})

In [10]:
rerank = CohereRerank(model="rerank-multilingual-v3.0",top_n=3)

compressor_retriver = ContextualCompressionRetriever(
    base_compressor=rerank,
    base_retriever=naive_retriever
)


In [11]:
TEMPLATE = """
    Você é um especialista em visão computacional para a parte de visão estéreo. Responda a pergunta utilizando o contexto informado
    Query:
    {question}

    Context:
    {context}
"""

rag_prompt = ChatPromptTemplate.from_template(TEMPLATE)

In [12]:
setup_retrival = RunnableParallel({"question":RunnablePassthrough(), "context":compressor_retriver})
output_parser = StrOutputParser()

compressor_retrieval_chain = setup_retrival | rag_prompt | llm | output_parser

In [13]:
compressor_retrieval_chain.invoke("Quais são os principais conceitos de visão estéreo?")

'\nNa visão estéreo, os principais conceitos incluem a correspondência estéreo, que busca encontrar correspondências entre pontos de uma imagem e sua correspondente na outra imagem capturada pelo segundo sensor. Além disso, a triangulação é um conceito essencial para determinar a profundidade dos objetos na cena com base nas correspondências estéreo. Outro conceito importante é a calibração estéreo, que envolve a determinação dos parâmetros intrínsecos e extrínsecos das câmeras, bem como a correção de distorções geométricas. A reconstrução tridimensional é outro conceito fundamental, que consiste em gerar um modelo tridimensional da cena a partir das informações estéreo obtidas.'